

This notebook contains the complete NLP pipeline:
1. Text Normalization
2. Regex Cleaning
3. Synonym Expansion
4. Tokenization + N-grams
5. Keyword Matching against Supabase DB
6. ML Model Prediction (from ml_trainer)
7. Final NLP Score = (keyword_score x 0.6) + (ml_score x 0.4)


In [1]:
# Run once
import subprocess
subprocess.run(["pip", "install", "nltk", "scikit-learn", "numpy", "gensim", "scipy"], check=True)


CompletedProcess(args=['pip', 'install', 'nltk', 'scikit-learn', 'numpy', 'gensim', 'scipy'], returncode=0)

In [2]:
import re
import string
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.util import ngrams

nltk.download('punkt',     quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("All libraries imported successfully")


All libraries imported successfully


In [12]:
SYNONYM_MAP = {
    "registration fee": [
        "joining fee", "processing fee", "training fee",
        "security deposit", "refundable deposit", "enrolment fee",
        "reg fee", "registration charge", "onboarding fee",
        "zero cost onboarding", "complimentary training",
        "admission fee", "application fee",
        # Expanded variations
        "joining charges", "joining charge", "joining fees",
        "registration charges", "registration charge", "registration fees",
        "processing charges", "processing charge", "processing fees",
        "training charges", "training charge", "training fees",
        "onboarding charges", "onboarding charge", "onboarding fees",
        "joining cost", "registration cost", "onboarding cost"
    ],
    "whatsapp hr": [
        "contact on whatsapp", "whatsapp only", "ping on whatsapp",
        "message on wa", "whatsapp number", "wa number",
        "contact via whatsapp", "whatsapp me", "msg on whatsapp",
        # Expanded variations
        "contact wa", "contact on wa", "contact via wa", "msg on wa", "ping on wa",
        "whatsapp group", "wa group"
    ],
    "instant joining": [
        "direct joining", "immediate joining", "join immediately",
        "join today", "start today", "joining today",
        "same day joining", "joining without interview"
    ],
    "no interview": [
        "no selection process", "instant selection", "no interview needed",
        "no interview required", "direct selection", "without interview",
        "no screening", "no test required"
    ],
    "earn daily": [
        "daily earnings", "earn per day", "daily income",
        "paid daily", "daily payment", "earn every day",
        "get paid daily", "daily payout"
    ],
    "limited seats": [
        "limited slots", "hurry up", "last few seats",
        "only few left", "seats filling fast", "apply now limited",
        "urgent requirement", "limited vacancy"
    ],
    "guaranteed job": [
        "100 percent placement", "job guaranteed", "assured placement",
        "placement guarantee", "guaranteed placement", "sure shot job",
        "confirmed placement", "100% job"
    ],
    "work from home earn": [
        "earn from home", "work at home", "home based job",
        "online earning", "earn sitting at home", "home job",
        "remote earning", "online income"
    ],
    "pay to work": [
        "pay and work", "invest to earn", "deposit to join",
        "pay before joining", "fee before work", "charges to join"
    ],
}

print(f"Total base keywords : {len(SYNONYM_MAP)}")
print(f"Total synonym variations : {sum(len(v) for v in SYNONYM_MAP.values())}")


Total base keywords : 9
Total synonym variations : 101


In [4]:
# These come from flagged_keywords table in Supabase
# Hardcoded here for testing — in production fetch from DB

FRAUD_KEYWORDS = {
    "pay to work":            0.98,
    "registration fee":       0.95,
    "training fee":           0.92,
    "whatsapp hr":            0.91,
    "deposit required":       0.90,
    "work from home earn":    0.85,
    "instant joining":        0.82,
    "guaranteed job":         0.80,
    "daily payment":          0.78,
    "earn daily":             0.77,
    "no interview":           0.75,
    "100% placement":         0.72,
    "limited seats":          0.69,
    "no experience required": 0.55,
    "urgent hiring":          0.50,
}

print(f"Total fraud keywords in database : {len(FRAUD_KEYWORDS)}")
print("\nKeywords sorted by fraud weight:")
for kw, wt in sorted(FRAUD_KEYWORDS.items(), key=lambda x: -x[1]):
    print(f"  {kw:<30} → {wt}")


Total fraud keywords in database : 15

Keywords sorted by fraud weight:
  pay to work                    → 0.98
  registration fee               → 0.95
  training fee                   → 0.92
  whatsapp hr                    → 0.91
  deposit required               → 0.9
  work from home earn            → 0.85
  instant joining                → 0.82
  guaranteed job                 → 0.8
  daily payment                  → 0.78
  earn daily                     → 0.77
  no interview                   → 0.75
  100% placement                 → 0.72
  limited seats                  → 0.69
  no experience required         → 0.55
  urgent hiring                  → 0.5


In [13]:
# Customize NLTK stopwords list to retain critical scam-signal words
EXCLUDED_STOP_WORDS = {
    "no", "not", "from", "to", "on", "me", "up", "few", "only", "now", "at", "and", "before", "without", "under"
}
STOP_WORDS = set(stopwords.words('english')) - EXCLUDED_STOP_WORDS

def normalize(text: str) -> str:
    """Step 1 — Lowercase, remove unicode/emoji, strip spaces"""
    text = str(text).lower()
    text = text.strip()
    text = text.encode('ascii', 'ignore').decode()
    return text

def clean(text: str) -> str:
    """Step 2 — Remove URLs, emails, symbols, fix spaces"""
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def expand_synonyms(text: str) -> str:
    """Step 3 — Replace synonym phrases with base keyword"""
    for base_keyword, synonyms in SYNONYM_MAP.items():
        for synonym in synonyms:
            text = text.replace(synonym, base_keyword)
    return text

def remove_stopwords(text: str) -> str:
    """Remove common English words that add no fraud signal"""
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return ' '.join(tokens)

def preprocess(text: str) -> str:
    """Full preprocessing pipeline"""
    text = normalize(text)
    text = clean(text)
    text = expand_synonyms(text)
    text = remove_stopwords(text)
    return text

# Test preprocessing
test = "Pay Registration Fee of Rs 500!! Instant Joining. No Interview Required."
print("Original  :", test)
print("After norm:", normalize(test))
print("After clean:", clean(normalize(test)))
print("After syn :", expand_synonyms(clean(normalize(test))))
print("Final     :", preprocess(test))


Original  : Pay Registration Fee of Rs 500!! Instant Joining. No Interview Required.
After norm: pay registration fee of rs 500!! instant joining. no interview required.
After clean: pay registration fee of rs 500 instant joining no interview required
After syn : pay registration fee of rs 500 instant joining no interview
Final     : pay registration fee rs 500 instant joining no interview


In [ ]:
# Tokenization + N-grams
def tokenize_and_ngrams(text: str) -> list:
    """Step 4 — Split text into words + 2-word + 3-word phrases"""
    tokens   = word_tokenize(text)
    unigrams = tokens
    bigrams  = [' '.join(g) for g in ngrams(tokens, 2)]
    trigrams = [' '.join(g) for g in ngrams(tokens, 3)]
    return unigrams + bigrams + trigrams

# Test
sample = preprocess("Pay registration fee Rs 500. WhatsApp HR now. No interview required.")
phrases = tokenize_and_ngrams(sample)

print(f"Total phrases generated : {len(phrases)}")
print("\nAll phrases:")
for p in phrases:
    print(f"  '{p}'")


Total phrases generated : 27

All phrases:
  'pay'
  'registration'
  'fee'
  'rs'
  '500'
  'whatsapp'
  'hr'
  'now'
  'no'
  'interview'
  'pay registration'
  'registration fee'
  'fee rs'
  'rs 500'
  '500 whatsapp'
  'whatsapp hr'
  'hr now'
  'now no'
  'no interview'
  'pay registration fee'
  'registration fee rs'
  'fee rs 500'
  'rs 500 whatsapp'
  '500 whatsapp hr'
  'whatsapp hr now'
  'hr now no'
  'now no interview'


In [16]:
def match_keywords(phrases: list) -> dict:
    """Step 5 — Match phrases against fraud keywords"""
    matched = {}
    for phrase in phrases:
        if phrase in FRAUD_KEYWORDS:
            matched[phrase] = FRAUD_KEYWORDS[phrase]
    return matched

# Test
sample = preprocess("Pay registration fee Rs 500. WhatsApp HR now. No interview required.")
phrases = tokenize_and_ngrams(sample)
matched = match_keywords(phrases)

print("Matched fraud keywords:")
for kw, wt in matched.items():
    print(f"  {kw:<30} → weight {wt}")


Matched fraud keywords:
  registration fee               → weight 0.95
  whatsapp hr                    → weight 0.91
  no interview                   → weight 0.75


In [17]:
def calculate_keyword_score(matched_keywords: dict) -> float:
    """Step 6 — Average matched weights + bonus for multiple matches"""
    if not matched_keywords:
        return 0.0

    weights     = list(matched_keywords.values())
    avg_weight  = sum(weights) / len(weights)
    match_bonus = min(len(weights) * 0.05, 0.20)  # max 20% bonus
    score       = min(avg_weight + match_bonus, 1.0)
    return round(score, 4)

# Test
matched = {"registration fee": 0.95, "whatsapp hr": 0.91, "no interview": 0.75}
score   = calculate_keyword_score(matched)

print(f"Matched keywords   : {list(matched.keys())}")
print(f"Individual weights : {list(matched.values())}")
print(f"Average weight     : {sum(matched.values())/len(matched.values()):.4f}")
print(f"Match bonus        : {min(len(matched)*0.05, 0.20):.4f}")
print(f"keyword_score      : {score}")


Matched keywords   : ['registration fee', 'whatsapp hr', 'no interview']
Individual weights : [0.95, 0.91, 0.75]
Average weight     : 0.8700
Match bonus        : 0.1500
keyword_score      : 1.0


In [ ]:
# This cell calls `predict()` from `ml.py` to get `ml_score`

import pickle
import os
from scipy.sparse import hstack, csr_matrix

MODEL_PATH = "models/scam_detector.pkl"
TFIDF_PATH = "models/tfidf_vectorizer.pkl"
W2V_PATH   = "models/word2vec_model.pkl"
W2V_SIZE   = 100

def predict_ml(description: str, salary: float = 0) -> dict:
    """
    Load trained models and predict scam probability
    Uses TF-IDF + Word2Vec + Salary flag combined
    """
    if not os.path.exists(MODEL_PATH):
        print("Model not found. Run ml_trainer.ipynb first.")
        return {"ml_scam_score": 0.0, "label": "Unknown", "confidence": "0%"}

    # Load saved models
    with open(MODEL_PATH, 'rb') as f: model     = pickle.load(f)
    with open(TFIDF_PATH, 'rb') as f: tfidf     = pickle.load(f)
    with open(W2V_PATH,   'rb') as f: w2v_model = pickle.load(f)

    # Preprocess
    processed = preprocess(description)
    tokens    = word_tokenize(processed)

    # TF-IDF vector
    tfidf_vec = tfidf.transform([processed])

    # Word2Vec vector — average all word vectors
    vectors = [w2v_model.wv[w] for w in tokens if w in w2v_model.wv]
    w2v_vec = np.mean(vectors, axis=0) if vectors else np.zeros(W2V_SIZE)
    w2v_sparse = csr_matrix(w2v_vec.reshape(1, -1))

    # Salary flag
    salary_flag = csr_matrix([[1 if salary > 50000 else 0]])

    # Combine all features
    combined = hstack([tfidf_vec, w2v_sparse, salary_flag])

    # Predict
    label    = model.predict(combined)[0]
    proba    = model.predict_proba(combined)[0]
    ml_score = round(float(proba[1]), 4)

    return {
        "ml_scam_score": ml_score,
        "label":         "Scam" if label == 1 else "Safe",
        "confidence":    f"{max(proba):.2%}"
    }

# Test
result = predict_ml("Pay registration fee Rs 500. Instant joining. No interview.", salary=80000)
print("ML Prediction Result:")
for k, v in result.items():
    print(f"  {k} : {v}")


ML Prediction Result:
  ml_scam_score : 0.098
  label : Safe
  confidence : 90.20%


In [19]:
def get_risk_level(score: float) -> str:
    if score == 0.0:   return "Unknown"
    elif score < 0.30: return "Safe"
    elif score < 0.50: return "Low Risk"
    elif score < 0.70: return "Medium Risk"
    elif score < 0.90: return "High Risk"
    else:              return "Confirmed Scam"

def analyze(description: str, salary: float = 0) -> dict:
    """
    MAIN METHOD — Full NLP pipeline
    
    Runs:
    1. Keyword matching  → keyword_score
    2. ML prediction     → ml_score
    3. Combines both     → final_nlp_score
    """
    if not description or len(description.strip()) < 10:
        return {"error": "Description too short"}

    # Step 1-6: Keyword pipeline
    processed        = preprocess(description)
    phrases          = tokenize_and_ngrams(processed)
    matched_keywords = match_keywords(phrases)
    keyword_score    = calculate_keyword_score(matched_keywords)

    # Step 7: ML prediction
    ml_result  = predict_ml(description, salary)
    ml_score   = ml_result["ml_scam_score"]

    # Step 8: Combine both scores
    # keyword_score catches KNOWN scam words
    # ml_score      catches UNKNOWN scam patterns
    final_nlp_score = round((keyword_score * 0.6) + (ml_score * 0.4), 4)
    risk_level      = get_risk_level(final_nlp_score)

    return {
        "keyword_score":     keyword_score,
        "ml_score":          ml_score,
        "final_nlp_score":   final_nlp_score,
        "risk_level":        risk_level,
        "matched_keywords":  matched_keywords,
        "total_matches":     len(matched_keywords),
        "ml_label":          ml_result["label"],
        "ml_confidence":     ml_result["confidence"],
    }

print("analyze() function ready")


analyze() function ready


In [20]:
test_cases = [
    ("Pay registration fee Rs 500. Instant joining. No interview. WhatsApp HR now!", 80000),
    ("Joining charges apply. Direct selection. Contact via WA.", 60000),
    ("Work with engineering team on real projects. Interview required.", 18000),
    ("Earn daily Rs 3000. No experience needed. Guaranteed job.", 90000),
    ("Zero cost onboarding. Complimentary training charges apply.", 55000),
    ("Official internship. Technical interview scheduled. Offer letter provided.", 12000),
]

print("=" * 65)
print("  NLP ENGINE TEST RESULTS")
print("=" * 65)

for desc, salary in test_cases:
    result = analyze(desc, salary)
    print(f"\nInput        : {desc[:55]}...")
    print(f"Salary       : Rs {salary:,}")
    print(f"Keyword Score: {result.get('keyword_score', 'N/A')}")
    print(f"ML Score     : {result.get('ml_score', 'N/A')}")
    print(f"Final NLP    : {result.get('final_nlp_score', 'N/A')}")
    print(f"Risk Level   : {result.get('risk_level', 'N/A')}")
    print(f"Keywords     : {list(result.get('matched_keywords', {}).keys())}")
    print("-" * 65)


  NLP ENGINE TEST RESULTS

Input        : Pay registration fee Rs 500. Instant joining. No interv...
Salary       : Rs 80,000
Keyword Score: 1.0
ML Score     : 0.098
Final NLP    : 0.6392
Risk Level   : Medium Risk
Keywords     : ['registration fee', 'instant joining', 'no interview', 'whatsapp hr']
-----------------------------------------------------------------

Input        : Joining charges apply. Direct selection. Contact via WA...
Salary       : Rs 60,000
Keyword Score: 1.0
ML Score     : 0.1904
Final NLP    : 0.6762
Risk Level   : Medium Risk
Keywords     : ['registration fee', 'no interview', 'whatsapp hr']
-----------------------------------------------------------------

Input        : Work with engineering team on real projects. Interview ...
Salary       : Rs 18,000
Keyword Score: 0.0
ML Score     : 0.0006
Final NLP    : 0.0002
Risk Level   : Safe
Keywords     : []
-----------------------------------------------------------------

Input        : Earn daily Rs 3000. No expe